# PII demo walkthrough

Run this notebook **as yourself** (a normal user, not the pipeline service principal). It shows what a human sees in each zone:

| Step | What you show | Zone |
| --- | --- | --- |
| 1 | The PII contract drives everything | `pii_governance` |
| 2 | Landing data is locked down by ABAC (secure by default) | `pii_raw`, `pii_vault` |
| 3 | The verification gate decides what may be published | `pii_governance.batch_control` |
| 4 | Analysts get tokens and ciphertext only, and can still join and count | `pii_analytics` |
| 5 | Only the service principal can re-identify or forget a person | `pii_vault` |

Prerequisite: `pii_setup_job` and `pii_main_job` have run (and optionally `pii_scenarios_job` for batch 2).

In [0]:
dbutils.widgets.text("catalog", "serverless_stable_yit814_catalog")
catalog = dbutils.widgets.get("catalog")
spark.sql(f"USE CATALOG {catalog}")
print(f"Running as: {spark.sql('SELECT current_user()').first()[0]}  |  catalog: {catalog}")

## Step 1: the PII contract (preventive control)

Every source column is declared here, with how it is normalized and protected. Columns that are **not** in the contract are routed to `_rescued_data` and quarantined by the raw pipeline. `class_tag` is the system governed tag applied to the raw column.

In [0]:
%sql
SELECT table_name, column_name, pii_type, class_tag, method, normalize_rule, key_or_vault
FROM pii_governance.pii_contract
WHERE active
ORDER BY table_name, method DESC, column_name

## Step 2: the restricted zone is secure by default

The Lakeflow Connect landing tables hold cleartext PII, and PII cannot be masked before ingestion. Two schema-level ABAC policies close that gap:

* a **column mask** on every column (`MATCH COLUMNS TRUE`, type-aware `system.data_classification.mask_value`)
* a **row filter** that returns no rows

Both apply `TO account users EXCEPT <pipeline service principal>` and have no `WHEN` clause, so every table that lands in the schema later is protected the moment it is created. You have `SELECT` on `pii_raw`, so ABAC (not a missing grant) is what hides the data.

In [0]:
%sql
-- Cleartext PII is in this table, but you see nothing
SELECT count(*) AS rows_visible_to_me FROM pii_raw.lfc_customers

In [0]:
%sql
SHOW EFFECTIVE POLICIES ON TABLE pii_raw.lfc_customers

In [0]:
%sql
DESCRIBE POLICY pii_raw_mask_all_columns ON SCHEMA pii_raw

In [0]:
%sql
-- Applied from the contract with system governed class.* tags (the same tags Data Classification uses)
SELECT table_name, column_name, tag_name
FROM information_schema.column_tags
WHERE schema_name = 'pii_raw'
ORDER BY table_name, column_name

In [0]:
%sql
-- Token -> value mapping. Same lockdown policies, and no grants for anyone but the service principal.
SELECT count(*) AS vault_rows_visible_to_me FROM pii_vault.tokmap_email

## Step 3: cleartext checks, quarantine and the verification gate

Format checks (email, E.164 phone, SSN) run on the **real, normalized** values inside the restricted zone, because once a value is a token it always looks valid. Failed rows go to `*_quarantine` (still in `pii_raw`, so you cannot see them either).

The verify pipeline then writes one row per table and batch with counts only, no values. The publish pipeline reads **only `VERIFIED` batches**. If you ran the scenarios job, batch 2 of `transactions` is `FAILED` on `no_cleartext_leak`: someone typed an email address into the `merchant` field, a column the contract says to keep.

In [0]:
%sql
SELECT table_name, batch_id, status, failed_checks,
       landed_rows, valid_rows, quarantined_rows, leak_rows, missing_token_rows, vault_conflicts, orphan_rows
FROM pii_governance.batch_control
ORDER BY table_name, batch_id

## Step 4: the pseudonymized zone

* `email`, `full_name`: **Pattern B, token vault**. A random `uuid` per normalized value, so the same person always gets the same token.
* `ssn`, `phone`, `dob`: **Pattern A, AES-256-GCM** with keys stored as Unity Catalog secrets (`*_enc`), plus a keyed **HMAC** (`*_hmac`). The ciphertext uses a random IV and can't be joined; the HMAC can.
* `street_address`, `card_number`: **dropped** (data minimization).

In [0]:
%sql
SELECT customer_id, full_name_token, email_token, ssn_enc, ssn_hmac, phone_enc, city, segment, signup_date, pii_key_version, _batch_id
FROM pii_analytics.customers
ORDER BY customer_id
LIMIT 10

In [0]:
%sql
-- Distinct counts and joins work on tokens and HMACs, without anyone re-identifying.
-- Here every row is a different person, so all counts match. The difference: encrypting the same SSN twice
-- gives a different ssn_enc (random IV), but always the same ssn_hmac, so only the HMAC can join across tables.
SELECT count(*)                    AS customers,
       count(DISTINCT email_token) AS distinct_people_by_email,
       count(DISTINCT ssn_hmac)    AS distinct_people_by_ssn,
       count(DISTINCT ssn_enc)     AS distinct_ciphertexts
FROM pii_analytics.customers

In [0]:
%sql
SELECT * FROM pii_analytics.customer_spend_by_segment ORDER BY segment, currency

In [0]:
%sql
-- Tags on raw tables do not carry over, so the published columns are tagged again (ungoverned tags)
SELECT column_name, tag_name, tag_value
FROM information_schema.column_tags
WHERE schema_name = 'pii_analytics' AND table_name = 'customers'
ORDER BY column_name, tag_name

## Step 5: re-identification and the right to be forgotten (service principal only)

* **Re-identify:** the `reidentify_check` task of `pii_main_job` (notebook `05_reidentify`) runs as the service principal. It decrypts with `pii_vault.pii_decrypt` and joins the vault, then checks that every published value round-trips to the original. It reports counts only.
* **Forget a person:** run as the service principal, delete their rows from the vault tables and the raw zone, then `VACUUM`. Their tokens in `pii_analytics` stay, but can no longer be reversed:

```sql
DELETE FROM pii_vault.tokmap_email WHERE token = '<email_token>';
DELETE FROM pii_vault.tokmap_name  WHERE token = '<full_name_token>';
```

* **Known limitation:** ABAC does not cover secrets or functions. Anyone with `ALL PRIVILEGES` on the catalog inherits `READ SECRET` on the keys. In production, put `pii_vault` in its own catalog that only the service principal can use.

# Part 2: under the hood (code and pipelines)

Everything is one Declarative Automation Bundle. Repo map:

| Path | What it is | Step |
| --- | --- | --- |
| `databricks.yml` | Bundle: variables, `run_as` service principal, `dev` target | |
| `resources/pii_jobs.job.yml` | `pii_setup_job`, `pii_main_job`, `pii_scenarios_job` | |
| `resources/pii_pipelines.pipeline.yml` | 3 serverless pipelines: raw, verify, publish | |
| `src/setup/00_schemas_contract` | Zones + grants, PII contract | 2a |
| `src/setup/01_secrets_udfs` | UC secrets, AES-GCM / HMAC UDFs, self-test | 4 |
| `src/setup/02_seed_landing` | Simulated Lakeflow Connect landing (batch 1, batch 2 scenarios) | 0 |
| `src/setup/03_tags_abac` | `class.*` column tags from the contract, secure-by-default policies | 2b |
| `src/pipelines/raw/utilities/pii_contract.py` | Contract-driven helpers: conform, normalize, protect | 1, 3b, 4 |
| `src/pipelines/raw/transformations/*.py` | Raw pipeline datasets (customers, transactions, vault) | 1-4 |
| `src/pipelines/verify/transformations/batch_control.py` | Verification gate | 5 |
| `src/pipelines/publish/transformations/analytics.py` | Publish verified batches | 6 |
| `src/notebooks/04_tag_outputs`, `05_reidentify` | Output tags, re-identification check | 6 |

## 2.1 Bundle and run identity

One top-level `run_as` makes **every job and pipeline run as the service principal**. You deploy the bundle, but you never run code in the restricted zone:

```yaml
variables:
  catalog:
    default: serverless_stable_yit814_catalog
  sp_application_id:
    default: 4ab6274e-d6e5-4839-a11a-04034a5d48de

run_as:
  service_principal_name: ${var.sp_application_id}
```

Why is setup a **job** and not bundle `schemas:` resources? When the deployer is not the `run_as` identity, a bundle can only manage jobs and pipelines, and a bundle schema is always owned by the deployer. Running setup as a job under the SP makes the SP the owner of the schemas, secrets, functions and policies.

The next cell reads the deployed jobs and pipelines from the workspace and shows the task graph and run identity.

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

print("JOBS (task <- depends on)")
for job in w.jobs.list(name=None):
    if not any(n in job.settings.name for n in ("pii_setup_job", "pii_main_job", "pii_scenarios_job")):
        continue
    full = w.jobs.get(job.job_id)
    print(f"\n{full.settings.name}  (run_as: {full.run_as_user_name})")
    for t in full.settings.tasks:
        kind = "pipeline" if t.pipeline_task else "notebook" if t.notebook_task else "run_job" if t.run_job_task else "other"
        deps = ", ".join(d.task_key for d in (t.depends_on or [])) or "-"
        print(f"  {t.task_key:<18} [{kind:<8}] <- {deps}")

print("\nPIPELINES")
for p in w.pipelines.list_pipelines(filter="name LIKE '%pii_%_pipeline%'"):
    spec = w.pipelines.get(p.pipeline_id).spec
    print(f"  {p.name:<40} schema={spec.schema:<15} channel={spec.channel:<8} run_as={p.run_as_user_name}")

## 2.2 Step 1 and 3b: contract-driven schema check and normalization

`utilities/pii_contract.py` reads the contract once when the pipeline graph is built. `conform()` is the schema gate: only contract columns pass, cast and normalized. **Anything undeclared is packed into `_rescued_data`**, and the row is then quarantined, so a new source column can never carry PII past pseudonymization.

```python
def conform(df, contract):
    undeclared = [c for c in df.columns if c not in contract and c not in LINEAGE_COLUMNS]
    if undeclared:
        any_value = F.coalesce(*[F.col(c).cast("string") for c in undeclared])
        rescued = F.when(any_value.isNotNull(), F.to_json(F.struct(*[F.col(c) for c in undeclared])))
    else:
        rescued = F.lit(None).cast("string")
    cols = []
    for name, spec in contract.items():
        expr = _normalize(name, spec["normalize_rule"]) if name in df.columns else F.lit(None)
        cols.append(expr.cast(spec["data_type"]).alias(name))
    ...
```

Normalization runs **before** tokenizing, so `"  ALICE.SMITH1@GMAIL.COM "` and `"alice.smith1@gmail.com"` get the same token. The rules come from `normalize_rule` in the contract:

| Rule | Implementation |
| --- | --- |
| `lower_trim` | `lower(trim(email))` |
| `e164` | digits only, then `+1` + 10 digits, or `+` + 11 digits starting with 1, otherwise NULL (fails DQ) |
| `digits_only` | `regexp_replace(ssn, '[^0-9]', '')` |
| `name` | trim, collapse spaces, `initcap` |

Batch 2 of the scenarios proves it: the 10 re-sent customers with messy emails resolve to the same 10 tokens, and the vault has 0 conflicts.

## 2.3 Step 3a: cleartext data quality with the quarantine pattern

The checks need the real values, so they run in the restricted zone, on two flows that read the same conformed view:

```python
RULES = {
    "customer_id_present": "customer_id IS NOT NULL",
    "name_present": "full_name IS NOT NULL AND full_name <> ''",
    "valid_email": "email RLIKE '^[^@ ]+@[^@ ]+[.][a-z]{2,}$'",
    "valid_phone_e164": "phone RLIKE '^[+][0-9]{11,15}$'",
    "valid_ssn": "length(ssn) = 9 AND ssn NOT LIKE '000%' AND substr(ssn, 4, 2) <> '00'",
    "no_rescued_data": "_rescued_data IS NULL",
}

@dp.table(name="customers_valid", table_properties={..., "pipelines.reset.allowed": "false"})
@dp.expect_all_or_drop({k: f"coalesce(({v}), false)" for k, v in RULES.items()})
def customers_valid():
    return spark.readStream.table("customers_conformed")

@dp.table(name="customers_quarantine")
def customers_quarantine():
    return (spark.readStream.table("customers_conformed")
            .where(f"NOT ({all_rules_pass(RULES)})")
            .withColumn("_failed_rules", failed_rules(RULES))   # rule names, never values
            .withColumn("_quarantined_at", F.current_timestamp()))
```

* `coalesce(..., false)` makes a NULL count as a failure, so every row lands in exactly one of the two tables. The gate later checks `landed = valid + quarantined`.
* The quarantine holds cleartext PII, so it stays in `pii_raw`, under the same lockdown.

## 2.4 Step 4, Pattern A: field-level encryption with Unity Catalog secrets

The keys are UC secrets in `pii_vault` (32 random bytes each, created by the SP, never printed). The encryption function **declares its secrets in a `SECRETS` clause**: it reads them with the function owner's (the SP's) permissions, so callers never touch key material.

```sql
CREATE OR REPLACE FUNCTION pii_vault.pii_protect(value STRING, key_name STRING)
RETURNS STRUCT<enc: STRING, hmac: STRING>
LANGUAGE PYTHON
SECRETS (pii_vault.ssn_key_v1, pii_vault.phone_key_v1, pii_vault.dob_key_v1, pii_vault.hmac_key_v1)
ENVIRONMENT (dependencies = '["cryptography"]', environment_version = '6')
AS $$
  key  = base64.b64decode(get(catalog=..., schema='pii_vault', key=key_name))
  hkey = base64.b64decode(get(catalog=..., schema='pii_vault', key='hmac_key_v1'))
  nonce = os.urandom(12)
  enc = base64.b64encode(nonce + AESGCM(key).encrypt(nonce, value.encode(), None))
  return {'enc': enc, 'hmac': hmac.new(hkey, value.encode(), hashlib.sha256).hexdigest()}
$$
```

* The ciphertext layout (12-byte IV, ciphertext, GCM tag) matches the built-in `aes_encrypt(..., 'GCM')`, so `try_aes_decrypt` can also read it given the key.
* There is a random IV per value, so the same SSN encrypts differently every time. The HMAC column is what you join and count on.
* `pii_key_version = 1` is stored on every row for key rotation.

Two platform details shaped this code:

| Finding | What we did |
| --- | --- |
| A query can call at most 5 UC Python UDFs (public preview). 3 encrypted columns x (encrypt + HMAC) = 6 | One `pii_protect` call returns both, kept as a struct and flattened in the next dataset so the optimizer never calls it twice |
| UDFs with `SECRETS` fail in a pipeline on the `CURRENT` channel (`SECRETS_NOT_ENABLED`) | The raw pipeline runs on the `PREVIEW` channel |

## 2.5 Step 4, Pattern B: the token vault

One streaming table per tokenized column, generated from the contract (`tokmap_email`, `tokmap_name`):

```python
@dp.table(name=f"{VAULT}.{vault_table}", table_properties={"pipelines.reset.allowed": "false"})
def _vault():
    return (spark.readStream.table("customers_valid")
            .select(F.col(column).alias("value"))
            .where("value IS NOT NULL")
            .dropDuplicates(["value"])                 # streaming state: a value gets a token once, ever
            .select(F.expr("uuid()").alias("token"), "value", F.current_timestamp().alias("created_at")))
```

Then `customers_pseudo` (a materialized view) joins the vault in and drops the cleartext columns:

```python
df = flatten_protected(spark.read.table("customers_protected"), CONTRACT)
for col, spec in TOKENIZE.items():
    tokens = spark.read.table(f"{VAULT}.{spec['key_or_vault']}").select(
        F.col("value").alias(col), F.col("token").alias(f"{col}_token"))
    df = df.join(tokens, col, "left").drop(col)
```

* `pipelines.reset.allowed=false` on the vault **and** on `customers_valid` (its source) means a full refresh can never regenerate tokens.
* The left join is on purpose: a missing token shows up as NULL and fails the gate, instead of silently dropping the row.
* The vault's table is published to `pii_vault` straight from a pipeline whose default schema is `pii_raw`, by using a fully qualified name.

## 2.6 Steps 5 and 6: the verification gate and publishing

`batch_control` (verify pipeline) computes one row per table and batch. It is the only place that compares the pseudonymized output with the cleartext source, which is allowed because the pipeline runs as the SP:

```python
GATE = {
    "counts_reconcile":              "landed_rows = valid_rows + quarantined_rows",
    "every_valid_row_pseudonymized": "candidate_rows = valid_rows",
    "every_pii_column_protected":    "unprotected_columns = 0",
    "no_null_on_value":              "null_on_value_rows = 0",
    "no_token_equals_source":        "token_equals_source_rows = 0",
    "every_token_resolves":          "missing_token_rows = 0",
    "vault_has_no_conflicts":        "vault_conflicts = 0",
    "no_cleartext_leak":             "leak_rows = 0",   # regex scan of every non-protected string column
}
```

The checks are warn-level expectations plus a `status` column, so one bad batch is held back without blocking the good ones. The publish pipeline takes only `VERIFIED` batches, and only the columns derived from the contract:

```python
def _verified(table, source):
    verified = (spark.read.table(f"{GOV}.batch_control")
                .where((F.col("table_name") == table) & (F.col("status") == "VERIFIED"))
                .select(F.col("batch_id").alias("_batch_id")))
    return (spark.read.table(f"{RAW}.{source}")
            .join(verified, "_batch_id", "left_semi")
            .select(*_allow_list(table)))   # keep columns + *_enc / *_hmac / *_token, nothing else
```

The three pipelines are separate tasks of `pii_main_job`, so publish never starts if the raw or verify pipeline fails.

## 2.7 Step 2: tags and secure-by-default policies (`03_tags_abac`)

The tags come from the contract, using system governed tags because the account's governed-tag quota was full:

```python
for r in contract_rows_with_class_tag:
    spark.sql(f"ALTER TABLE {RAW}.lfc_{r.table_name} ALTER COLUMN `{r.column_name}` SET TAGS ('{r.class_tag}')")
```

The policies have no `WHEN` clause and use `MATCH COLUMNS TRUE`, so they protect every column of every current and future table, with no control tag needed:

```sql
CREATE OR REPLACE POLICY pii_raw_mask_all_columns ON SCHEMA pii_raw
COLUMN MASK system.data_classification.mask_value
TO `account users` EXCEPT `<sp>`
FOR TABLES MATCH COLUMNS TRUE AS c ON COLUMN c;

CREATE OR REPLACE POLICY pii_raw_deny_all_rows ON SCHEMA pii_raw
ROW FILTER pii_governance.pii_deny_all
TO `account users` EXCEPT `<sp>`
FOR TABLES;
```

The notebook takes `<sp>` from `current_user()` and refuses to run if it looks like a human (contains `@`), so a person can never be put in `EXCEPT` by mistake. The same pair of policies protects `pii_vault`.